# Bento's MNIST speed run


This notebook trains a small convolutional network on the 60,000 MNIST train images and tests it on the 10,000 test images, on one GPU, and measures the total time of training plus testing.


Run the cells from top to bottom.

At the end it prints the mean time and the mean test accuracy of 30 runs, each with ± the standard error of the mean.

Currently, on a google Colab, T4 GPU, the accuracy and run-time are

99.585 +- 0.008 % and  239.27 +- 2.44 ms respectively.

This as apposed to the original code [https://github.com/tuomaso/train_mnist_fast] for which the accuracy and run-time are

98.969 +- 0.058 % and 623 ms ± 8.43 ms  


# The code was developed under the following rules

| | Rule | Where to check it in this notebook |
|---|---|---|
| 1 | The accuracy is measured on all 10,000 test images. | See `accuracy_in_percent`  |
| 2 | Everything that is done to the data counts in the time, with three exceptions that stay outside the clock: loading the data, moving it to the GPU, and normalising it. | There is an "outside the clock" label to identify where this happens. Everything else that touches an image (conversion to half precision, the blank border, reading patches, shifting) happens inside `train_the_network` or `predict_the_digits`, which run inside the clock |
| 3 | The time is the total time of training plus testing, for the whole system. A reported number is the mean of 30 runs, after one warm-up run that is not counted. | Section 14 shows where the clock starts and stops. Section 15 does the warm-up run and the 30 runs. |
| 4 | With fixed settings, the test set never decides anything and never changes a weight.  | `train_the_network` (Section 11) receives only the train images and the train labels. The test images go only into `predict_the_digits` (section 12), which changes no weight. The test labels go only into `accuracy_in_percent`. |
| 5 | No pre-training. The initial weights are random and fresh in every run. | `create_random_initial_weights_in_one_vector` (section 6) has no input and is called inside every run. No random seed is set anywhere. |
| 6 | The method must work with a single test image: nothing may be based on several test images at once. | `predict_the_digits` (section 12) treats every image on its own; the text above it explains why. |
| 7 | A table built before the clock must come from a formula with a handful of settings. A table whose entries were tuned one by one would be learned weights and is forbidden. Building such tables inside the clock is preferred. | All four tables are built inside the clock, in every run, by functions that have no input (section 5). The only list prepared before the clock is the list of the nine shifts in section 2, which follows from one setting (`LARGEST_SHIFT_IN_PIXELS`). |
| 8 | The architecture is free. | Section 7. |

## Table 2. This solution next to the original solution

The original is the notebook  [https://github.com/tuomaso/train_mnist_fast].

| | Original | This notebook |
|---|---|---|
| Network | convolution 5x5, 24 channels (28x28 to 24x24), max-pooling 2x2 (to 12x12); convolution 3x3, 32 channels (to 10x10), max-pooling 2x2 (to 5x5); linear 800 to 256; linear 256 to 10. 215,194 weights. | convolution 5x5 with stride 2, 32 channels (28x28 to 12x12); convolution 5x5, 32 channels (to 8x8), max-pooling 2x2 (to 4x4); linear 512 to 256; linear 256 to 10. 160,362 weights. |
| Activation | ReLU | GELU |
| Precision | 32-bit | The network computes in 16-bit (half precision). The weights are kept and updated in 32-bit, and the loss is computed in 32-bit. |
| Initial weights | PyTorch's default random initialisation | the same |
| Batch size | 256 | 384 |
| Passes over the train images | 1 pass (235 steps) | 1 pass over all images (157 steps), then 78 steps on hard images only, which is half a pass counted in images (235 steps in total) |
| Optimizer and learning rate | `torch.optim.Adam`, betas 0.7 and 0.9. Learning rate by `OneCycleLR`: from 0.0004 up to 0.01 during the first 30% of the steps, then down to almost zero, along cosine curves. | Adam written by hand (same arithmetic), betas 0.8 and 0.85. Learning rate from 0.0004 up to 0.01 during the first 25% of the steps, then down to almost zero, along straight lines. |
| Loss | cross-entropy | cross-entropy with label smoothing 0.2 |
| Images trained on after the first pass | none (there is only one pass) | only the hard images: from each batch of the first pass, the 38 images (10%) with the largest loss, 5,928 images in all, in random order |
| Augmentation | none | none in the first pass; after it, each batch is shifted at random by -1, 0 or +1 pixel in each direction |
| Test-time method | one prediction per test image | one prediction per test image; an image whose own two largest probabilities differ by less than 0.5 gets a second look through 8 shifted windows (about 270 of the 10,000 images per run, by the project's record) |
| Code-level speed tricks | none: `nn.Conv2d`, `nn.Linear`, `torch.optim.Adam`, `OneCycleLR` | half precision; first convolution as "read every patch through a list of positions, then one matrix product"; all weights in one vector; Adam factors and smoothed targets read from small tables; one training step recorded as a CUDA graph and replayed (sections 4 to 9; table 3 at the end says what each is worth) |
| Total time (train + test) | 0.623 s  | **0.23927 s** , 2.6 times faster |
| Accuracy on the 10,000 test images | 98.969 % | **99.576%** , +0.607 |
| How it was measured | mean of 30 runs on Tesla T4, PyTorch 2.14.1+cu130, as recorded by the project | mean of 30 runs after one warm-up run. These are the numbers printed in section 15 of this file. |

## Imports

In [1]:
import gc
import math
import platform
import time
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torchvision import datasets

## 1. All the settings

Every number that can be changed is here, and nowhere below. In order: where the data and the GPU are; facts about MNIST; the network; the training; the learning rate and Adam; one setting of the recording (section 9); the fixed threshold of the test (section 12); the number of runs.

In [2]:
FOLDER_WITH_THE_MNIST_DATA = 'data'
THE_GPU = torch.device('cuda')

IMAGE_SIDE = 28
NUMBER_OF_DIGITS = 10
NUMBER_OF_TRAIN_IMAGES = 60000
LARGEST_PIXEL_VALUE_IN_THE_FILES = 255.0

FIRST_CONVOLUTION_KERNEL = 5
FIRST_CONVOLUTION_STRIDE = 2
FIRST_CONVOLUTION_CHANNELS = 32
SECOND_CONVOLUTION_KERNEL = 5
SECOND_CONVOLUTION_CHANNELS = 32
POOLING_SIZE = 2
HIDDEN_LAYER_SIZE = 256

BATCH_SIZE = 384
PASSES_AFTER_THE_FIRST_PASS = 0.5
FRACTION_OF_EACH_BATCH_KEPT_AS_HARD_IMAGES = 0.1
LARGEST_SHIFT_IN_PIXELS = 1
LABEL_SMOOTHING = 0.2

PEAK_LEARNING_RATE = 0.01
PEAK_DIVIDED_BY_STARTING_LEARNING_RATE = 25.0
STARTING_DIVIDED_BY_FINAL_LEARNING_RATE = 1e4
FRACTION_OF_THE_STEPS_WITH_RISING_LEARNING_RATE = 0.25
ADAM_BETA_1 = 0.8
ADAM_BETA_2 = 0.85
ADAM_EPSILON = 1e-8

STEPS_RUN_DIRECTLY_BEFORE_THE_RECORDING = 2

GAP_BELOW_WHICH_AN_IMAGE_GETS_A_SECOND_LOOK = 0.5

NUMBER_OF_WARM_UP_RUNS_NOT_COUNTED = 1
NUMBER_OF_TIMED_RUNS = 30

## 2. Sizes that follow from the settings

Nothing can be chosen here: these are the sizes of the layers, the number of steps, and the list of the nine windows, computed from the settings above.

- An image is stored with a blank border of one pixel (30x30), so that a window shifted by one pixel still finds something to read.
- The weight tensors are listed with their shape and with the number of inputs of their layer (used for the random initialisation).
- Window 0 is not shifted. Windows 1 to 8 are shifted by one pixel in the eight directions.

In [3]:
STORED_IMAGE_SIDE = IMAGE_SIDE + 2 * LARGEST_SHIFT_IN_PIXELS
PIXELS_IN_A_STORED_IMAGE = STORED_IMAGE_SIDE ** 2

PIXELS_IN_A_PATCH = FIRST_CONVOLUTION_KERNEL ** 2
SIDE_AFTER_FIRST_CONVOLUTION = (
    (IMAGE_SIDE - FIRST_CONVOLUTION_KERNEL) // FIRST_CONVOLUTION_STRIDE + 1)
SIDE_AFTER_SECOND_CONVOLUTION = SIDE_AFTER_FIRST_CONVOLUTION - SECOND_CONVOLUTION_KERNEL + 1
SIDE_AFTER_POOLING = SIDE_AFTER_SECOND_CONVOLUTION // POOLING_SIZE
NUMBER_OF_FEATURES = SIDE_AFTER_POOLING ** 2 * SECOND_CONVOLUTION_CHANNELS
INPUTS_OF_THE_SECOND_CONVOLUTION = SECOND_CONVOLUTION_KERNEL ** 2 * FIRST_CONVOLUTION_CHANNELS

SHAPE_OF_EACH_WEIGHT_TENSOR = {
    'first convolution weights': (PIXELS_IN_A_PATCH, FIRST_CONVOLUTION_CHANNELS),
    'first convolution biases': (FIRST_CONVOLUTION_CHANNELS,),
    'second convolution weights': (SECOND_CONVOLUTION_CHANNELS, SECOND_CONVOLUTION_KERNEL,
                                   SECOND_CONVOLUTION_KERNEL, FIRST_CONVOLUTION_CHANNELS),
    'second convolution biases': (SECOND_CONVOLUTION_CHANNELS,),
    'hidden layer weights': (NUMBER_OF_FEATURES, HIDDEN_LAYER_SIZE),
    'hidden layer biases': (HIDDEN_LAYER_SIZE,),
    'output layer weights': (HIDDEN_LAYER_SIZE, NUMBER_OF_DIGITS),
    'output layer biases': (NUMBER_OF_DIGITS,),
}
NUMBER_OF_INPUTS_OF_EACH_WEIGHT_TENSOR = {
    'first convolution weights': PIXELS_IN_A_PATCH,
    'first convolution biases': PIXELS_IN_A_PATCH,
    'second convolution weights': INPUTS_OF_THE_SECOND_CONVOLUTION,
    'second convolution biases': INPUTS_OF_THE_SECOND_CONVOLUTION,
    'hidden layer weights': NUMBER_OF_FEATURES,
    'hidden layer biases': NUMBER_OF_FEATURES,
    'output layer weights': HIDDEN_LAYER_SIZE,
    'output layer biases': HIDDEN_LAYER_SIZE,
}
SIZE_OF_EACH_WEIGHT_TENSOR = [
    math.prod(shape) for shape in SHAPE_OF_EACH_WEIGHT_TENSOR.values()]

STEPS_OF_THE_FIRST_PASS = math.ceil(NUMBER_OF_TRAIN_IMAGES / BATCH_SIZE)
IMAGES_READ_AFTER_THE_FIRST_PASS = PASSES_AFTER_THE_FIRST_PASS * NUMBER_OF_TRAIN_IMAGES
STEPS_ON_THE_HARD_IMAGES = round(IMAGES_READ_AFTER_THE_FIRST_PASS / BATCH_SIZE)
NUMBER_OF_STEPS = STEPS_OF_THE_FIRST_PASS + STEPS_ON_THE_HARD_IMAGES
FULL_BATCHES_OF_THE_FIRST_PASS = NUMBER_OF_TRAIN_IMAGES // BATCH_SIZE
HARD_IMAGES_KEPT_FROM_EACH_BATCH = round(
    FRACTION_OF_EACH_BATCH_KEPT_AS_HARD_IMAGES * BATCH_SIZE)

POSSIBLE_SHIFTS = range(-LARGEST_SHIFT_IN_PIXELS, LARGEST_SHIFT_IN_PIXELS + 1)
SHIFT_OF_EACH_WINDOW = [(0, 0)]
for shift_down in POSSIBLE_SHIFTS:
    for shift_right in POSSIBLE_SHIFTS:
        if (shift_down, shift_right) != (0, 0):
            SHIFT_OF_EACH_WINDOW.append((shift_down, shift_right))
NUMBER_OF_WINDOWS = len(SHIFT_OF_EACH_WINDOW)
WINDOW_WITHOUT_SHIFT = 0
SHIFTED_WINDOWS = range(1, NUMBER_OF_WINDOWS)

print(f'{sum(SIZE_OF_EACH_WEIGHT_TENSOR):,} weights')
print(f'{STEPS_OF_THE_FIRST_PASS} steps in the first pass + {STEPS_ON_THE_HARD_IMAGES} steps '
      f'on hard images = {NUMBER_OF_STEPS} steps of {BATCH_SIZE} images')
print(f'{HARD_IMAGES_KEPT_FROM_EACH_BATCH} hard images kept from each of the '
      f'{FULL_BATCHES_OF_THE_FIRST_PASS} full batches of the first pass')
print(f'{NUMBER_OF_WINDOWS} windows: 1 without shift + {len(SHIFTED_WINDOWS)} shifted')

160,362 weights
157 steps in the first pass + 78 steps on hard images = 235 steps of 384 images
38 hard images kept from each of the 156 full batches of the first pass
9 windows: 1 without shift + 8 shifted


## 3. Computations outside the clock: load MNIST, normalise it, move it to the GPU (rule 2)

This is the only work on the data that is not timed, and it is the same as in the original notebook: the pixels are divided by 255, then the mean of the train pixels is subtracted and the result is divided by their spread.

`VALUE_OF_A_BLANK_PIXEL_AFTER_NORMALISATION` is the value that a black pixel (0 in the file) has after this normalisation. It is one number that follows from the normalisation; it is used in section 4 to fill the border around the images.

In [4]:
def load_mnist_normalise_it_and_move_it_to_the_gpu_outside_the_clock():
    train_set = datasets.MNIST(root=FOLDER_WITH_THE_MNIST_DATA, train=True, download=True)
    test_set = datasets.MNIST(root=FOLDER_WITH_THE_MNIST_DATA, train=False, download=True)

    raw_train_images = train_set.data.float() / LARGEST_PIXEL_VALUE_IN_THE_FILES
    mean_of_the_train_pixels = raw_train_images.mean()
    spread_of_the_train_pixels = raw_train_images.std()

    train_images = raw_train_images.to(THE_GPU)
    train_images = (train_images - mean_of_the_train_pixels) / spread_of_the_train_pixels
    train_labels = train_set.targets.to(THE_GPU)

    test_images = test_set.data.float().to(THE_GPU) / LARGEST_PIXEL_VALUE_IN_THE_FILES
    test_images = (test_images - mean_of_the_train_pixels) / spread_of_the_train_pixels
    test_labels = test_set.targets.to(THE_GPU)

    value_of_a_blank_pixel = float(-mean_of_the_train_pixels / spread_of_the_train_pixels)
    return train_images, train_labels, test_images, test_labels, value_of_a_blank_pixel


(train_images, train_labels, test_images, test_labels,
 VALUE_OF_A_BLANK_PIXEL_AFTER_NORMALISATION) = (
    load_mnist_normalise_it_and_move_it_to_the_gpu_outside_the_clock())

assert len(train_images) == NUMBER_OF_TRAIN_IMAGES
print('train images', tuple(train_images.shape), '  test images', tuple(test_images.shape))

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 400kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.79MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.1MB/s]


train images (60000, 28, 28)   test images (10000, 28, 28)


## 4. Computations inside the clock: images in half precision, with a blank border

**Speed trick: half precision.** The images are converted to 16-bit numbers and the network computes with 16-bit numbers; the GPU is about twice as fast with them.

**Why no rule is broken.** The conversion and the border are work on the data, so they are inside the clock: this function is called by `train_the_network` (for the train images) and by `predict_the_digits` (for the images to predict). It treats every image separately and in the same way.

In [5]:
def store_images_in_half_precision_with_a_blank_border(images):
    border = LARGEST_SHIFT_IN_PIXELS
    images = images.half()
    images = F.pad(images, (border, border, border, border),
                   value=VALUE_OF_A_BLANK_PIXEL_AFTER_NORMALISATION)
    return images.reshape(-1, PIXELS_IN_A_STORED_IMAGE)

## 5. More computations inside the clock: four small tables built from the settings only (rule 7)

Each function below has no input, so a table can contain nothing but what a formula makes of the settings: nothing learned, nothing from the data. All four are built inside the clock, in every run.

**Table of pixel positions (a speed trick).** The first convolution looks at 12 x 12 = 144 patches of 5 x 5 pixels. For each patch we write down the positions (0 to 899) of its 25 pixels in the stored 30x30 image: one list of 3,600 positions. Reading a stored image at these positions gives all its patches at once (section 7 turns them into the convolution with one matrix product). A shifted window is the same list with one number added to every position. The table has nine such lists: row 0 without shift, rows 1 to 8 for the eight shifts by one pixel.

**Table of smoothed targets (a speed trick).** Row d is the target for the digit d: 0.82 for d itself and 0.02 for each other digit (label smoothing 0.2). Reading a row is cheaper than asking PyTorch to smooth the labels at every step.

**Table of Adam step sizes (a speed trick).** The learning rate of each step comes from a formula: it rises in a straight line from 0.0004 to 0.01 during the first 25% of the steps, then falls in a straight line to almost zero. Adam also divides its two running means by correction factors that depend only on the number of the step. Both are multiplied into one table: step size = learning rate / (1 - beta1^t) x sqrt(1 - beta2^t), and epsilon x sqrt(1 - beta2^t). This gives exactly the update of the textbook Adam, with fewer operations per step.

**Table of the first pass.** Step s of the first pass reads the train images 384 x s to 384 x s + 383, in the order in which they are stored. The last step would run past the end, so it starts at image 60,000 - 384 instead: every batch then has exactly 384 images, which the recorded step of section 9 needs.

In [6]:
def pixel_positions_read_through_each_window():
    patch_starts = torch.arange(SIDE_AFTER_FIRST_CONVOLUTION, device=THE_GPU)
    patch_starts = patch_starts * FIRST_CONVOLUTION_STRIDE
    steps_inside_a_patch = torch.arange(FIRST_CONVOLUTION_KERNEL, device=THE_GPU)

    rows = patch_starts.view(-1, 1, 1, 1) + steps_inside_a_patch.view(1, 1, -1, 1)
    columns = patch_starts.view(1, -1, 1, 1) + steps_inside_a_patch.view(1, 1, 1, -1)
    positions_from_the_corner = (rows * STORED_IMAGE_SIDE + columns).reshape(1, -1)

    corner_of_each_window = []
    for shift_down, shift_right in SHIFT_OF_EACH_WINDOW:
        corner_row = LARGEST_SHIFT_IN_PIXELS + shift_down
        corner_column = LARGEST_SHIFT_IN_PIXELS + shift_right
        corner_of_each_window.append(corner_row * STORED_IMAGE_SIDE + corner_column)
    corner_of_each_window = torch.tensor(corner_of_each_window, device=THE_GPU)

    return positions_from_the_corner + corner_of_each_window.view(-1, 1)


def smoothed_targets_of_each_digit():
    one_for_the_true_digit = torch.eye(NUMBER_OF_DIGITS, device=THE_GPU)
    share_of_every_digit = LABEL_SMOOTHING / NUMBER_OF_DIGITS
    return one_for_the_true_digit * (1 - LABEL_SMOOTHING) + share_of_every_digit


def adam_step_size_and_adam_epsilon_of_each_step():
    peak = PEAK_LEARNING_RATE
    start = peak / PEAK_DIVIDED_BY_STARTING_LEARNING_RATE
    final = start / STARTING_DIVIDED_BY_FINAL_LEARNING_RATE
    last_rising_step = FRACTION_OF_THE_STEPS_WITH_RISING_LEARNING_RATE * NUMBER_OF_STEPS - 1
    falling_steps = NUMBER_OF_STEPS - 1 - last_rising_step

    step = torch.arange(NUMBER_OF_STEPS, device=THE_GPU, dtype=torch.float32)
    rising = start + (peak - start) * step / last_rising_step
    falling = peak + (final - peak) * (step - last_rising_step) / falling_steps
    learning_rate = torch.where(step <= last_rising_step, rising, falling)

    correction_of_the_mean = 1 - ADAM_BETA_1 ** (step + 1)
    correction_of_the_mean_square = (1 - ADAM_BETA_2 ** (step + 1)).sqrt()

    adam_step_size = learning_rate / correction_of_the_mean * correction_of_the_mean_square
    adam_epsilon = ADAM_EPSILON * correction_of_the_mean_square
    return adam_step_size, adam_epsilon


def train_images_of_each_step_of_the_first_pass():
    number_of_each_step = torch.arange(STEPS_OF_THE_FIRST_PASS, device=THE_GPU)
    last_possible_first_image = NUMBER_OF_TRAIN_IMAGES - BATCH_SIZE
    first_image_of_each_step = number_of_each_step * BATCH_SIZE
    first_image_of_each_step = first_image_of_each_step.clamp(max=last_possible_first_image)

    place_in_the_batch = torch.arange(BATCH_SIZE, device=THE_GPU)
    return first_image_of_each_step.view(-1, 1) + place_in_the_batch.view(1, -1)


def build_the_tables_from_the_settings_only():
    tables = SimpleNamespace()
    tables.pixel_positions_of_each_window = pixel_positions_read_through_each_window()
    tables.smoothed_targets_of_each_digit = smoothed_targets_of_each_digit()
    (tables.adam_step_size_of_each_step,
     tables.adam_epsilon_of_each_step) = adam_step_size_and_adam_epsilon_of_each_step()
    tables.train_images_of_each_first_pass_step = train_images_of_each_step_of_the_first_pass()
    return tables

## 6. Even more computations inside the clock: random initial weights and empty storage (rule 5)

**The initial weights are random**, drawn fresh in every run with PyTorch's default rule for `nn.Conv2d` and `nn.Linear`: uniform between -1/sqrt(n) and +1/sqrt(n), where n is the number of inputs of the layer. The function has no input and no seed is set.

**Speed trick: all weights in one vector.** The 160,362 weights of the four layers sit one after the other in one vector. One conversion to half precision and one Adam update then serve all layers at once, instead of eight of each.

**The empty storage** is what one training needs besides the weights: the two running means of Adam (zeros), the number of the current step (kept on the GPU, see section 8), the loss of each train image (filled during the first pass), and, for every step, which train images it reads and through which window (filled in sections 10 and 11).

Both are created inside the clock. The original code builds its model before the clock; doing it inside is stricter and costs nothing measurable (table 3).

In [7]:
def create_random_initial_weights_in_one_vector():
    pieces = []
    for name, size in zip(SHAPE_OF_EACH_WEIGHT_TENSOR, SIZE_OF_EACH_WEIGHT_TENSOR):
        bound = 1 / math.sqrt(NUMBER_OF_INPUTS_OF_EACH_WEIGHT_TENSOR[name])
        piece = torch.empty(size, device=THE_GPU).uniform_(-bound, bound)
        pieces.append(piece)
    return torch.cat(pieces)


def create_the_random_initial_weights_and_the_empty_storage():
    storage = SimpleNamespace()
    storage.all_weights = create_random_initial_weights_in_one_vector().requires_grad_(True)

    storage.adam_mean_of_the_gradients = torch.zeros_like(storage.all_weights)
    storage.adam_mean_of_the_squared_gradients = torch.zeros_like(storage.all_weights)
    storage.number_of_the_step = torch.zeros(1, dtype=torch.long, device=THE_GPU)

    storage.loss_of_each_train_image = torch.zeros(NUMBER_OF_TRAIN_IMAGES, device=THE_GPU)
    storage.train_images_of_each_step = torch.zeros(
        NUMBER_OF_STEPS, BATCH_SIZE, dtype=torch.long, device=THE_GPU)
    storage.window_of_each_step = torch.zeros(NUMBER_OF_STEPS, dtype=torch.long, device=THE_GPU)

    return storage

## 7. The architecture

| Layer | From | To |
|---|---|---|
| first convolution, 5x5, stride 2, 32 channels, then GELU | 28x28 image | 12x12 with 32 channels |
| second convolution, 5x5, 32 channels | | 8x8 with 32 channels |
| max-pooling 2x2, then GELU | | 4x4 with 32 channels = 512 numbers |
| hidden linear layer, then GELU | 512 | 256 |
| output linear layer | 256 | 10 outputs, one per digit |

- `split_the_weight_vector_into_the_tensors_of_the_layers` cuts the one vector into the eight tensors of section 2. It copies nothing.
- **Speed trick: the first convolution as one matrix product.** `patches` holds, for every image, the pixels of its 144 patches (read through the position lists of section 5). Multiplying the (144 x images) rows of 25 pixels by the 25 x 32 weight matrix is the same arithmetic as `nn.Conv2d(1, 32, 5, stride=2)`.
- **Speed trick: the channel is kept as the last axis.** The weights of the second convolution are stored as (channels out, 5, 5, channels in) and the image as (12, 12, channels). `permute` only tells PyTorch's functions where the channel axis is; it moves no number. In this layout the GPU's convolution is faster in half precision.
- **Speed trick: `torch.addmm(biases, x, weights)`** is `x @ weights + biases` in one operation, with the weights stored as (inputs, outputs).
- The network computes in half precision; its 10 outputs are converted to 32-bit at the end.

Nothing in the network looks at more than one image at a time: there is no normalisation layer, and every operation treats the images of a batch separately.

In [8]:
def split_the_weight_vector_into_the_tensors_of_the_layers(all_weights):
    pieces = all_weights.split(SIZE_OF_EACH_WEIGHT_TENSOR)
    shapes = SHAPE_OF_EACH_WEIGHT_TENSOR.values()
    return [piece.view(shape) for piece, shape in zip(pieces, shapes)]


def first_convolution_as_one_matrix_product(patches, weights, biases):
    number_of_images = patches.shape[0]
    side = SIDE_AFTER_FIRST_CONVOLUTION

    x = patches.view(-1, PIXELS_IN_A_PATCH) @ weights + biases
    x = F.gelu(x)
    return x.view(number_of_images, side, side, FIRST_CONVOLUTION_CHANNELS)


def second_convolution_then_max_pooling(x, weights, biases):
    x = x.permute(0, 3, 1, 2)
    kernels = weights.permute(0, 3, 1, 2)

    x = F.conv2d(x, kernels, biases)
    x = F.max_pool2d(x, POOLING_SIZE)
    x = F.gelu(x)
    return x.permute(0, 2, 3, 1)


def outputs_of_the_network(patches, all_weights_in_half_precision):
    (first_convolution_weights, first_convolution_biases,
     second_convolution_weights, second_convolution_biases,
     hidden_layer_weights, hidden_layer_biases,
     output_layer_weights, output_layer_biases) = (
        split_the_weight_vector_into_the_tensors_of_the_layers(all_weights_in_half_precision))

    x = first_convolution_as_one_matrix_product(
        patches, first_convolution_weights, first_convolution_biases)
    x = second_convolution_then_max_pooling(
        x, second_convolution_weights, second_convolution_biases)

    x = x.reshape(-1, NUMBER_OF_FEATURES)
    x = F.gelu(torch.addmm(hidden_layer_biases, x, hidden_layer_weights))
    x = torch.addmm(output_layer_biases, x, output_layer_weights)
    return x.float()

## 8. One training step

A step does what any training step does: read a batch, compute the outputs and the loss, compute the gradient, update the weights with Adam.

It is written in one particular way: the number of the step is a number on the GPU, and the step finds everything it needs through it, in tables: which train images to read, through which window, and the Adam step size. So the step contains no decision taken in Python, and the very same list of GPU operations serves for every step. That is what allows section 9 to record it once and replay it.

- The batch is read in two moves: the stored images of the batch, then their pixels at the positions of the window of this step. In the first pass the window is always the one without shift.
- The loss of every image of the batch is stored (rule 4: these are train images). Section 10 uses these losses.
- **Speed trick: Adam written by hand.** `adam_update_of_all_the_weights` is the textbook Adam on the one vector of weights: the running mean of the gradients, the running mean of their squares, and the update, with the step size and the epsilon of this step read from the table of section 5.

The step receives the train images and the train labels, and nothing of the test set.

In [9]:
def adam_update_of_all_the_weights(storage, tables, gradient):
    step = storage.number_of_the_step
    mean = storage.adam_mean_of_the_gradients
    mean_square = storage.adam_mean_of_the_squared_gradients

    mean.lerp_(gradient, 1 - ADAM_BETA_1)
    mean_square.mul_(ADAM_BETA_2).addcmul_(gradient, gradient, value=1 - ADAM_BETA_2)

    numerator = mean * tables.adam_step_size_of_each_step[step]
    denominator = mean_square.sqrt().add_(tables.adam_epsilon_of_each_step[step])
    storage.all_weights.addcdiv_(numerator, denominator, value=-1)


def one_training_step(storage, tables, stored_train_images, train_labels):
    step = storage.number_of_the_step
    images_of_the_batch = storage.train_images_of_each_step[step].view(BATCH_SIZE)
    window = storage.window_of_each_step[step]
    pixel_positions = tables.pixel_positions_of_each_window[window].view(-1)

    patches = stored_train_images[images_of_the_batch][:, pixel_positions]
    labels = train_labels[images_of_the_batch]
    targets = tables.smoothed_targets_of_each_digit[labels]

    outputs = outputs_of_the_network(patches, storage.all_weights.half())
    loss_of_each_image = F.cross_entropy(outputs, targets, reduction='none')

    with torch.no_grad():
        storage.loss_of_each_train_image[images_of_the_batch] = loss_of_each_image

    (gradient,) = torch.autograd.grad(loss_of_each_image.mean(), storage.all_weights)

    with torch.no_grad():
        adam_update_of_all_the_weights(storage, tables, gradient)
        storage.number_of_the_step += 1

## 9. Speed trick: one training step recorded once and replayed (CUDA graph)

**What it does.** Normally PyTorch sends the GPU the operations of a training step one by one, about a hundred small commands per step, and for a network this small the sending takes longer than the computing. A CUDA graph is a recording of the commands of one step. `recording.replay()` then runs the whole step with a single call.

**How it is used.** In every run, inside the clock, the first 2 steps are run in the ordinary way; then one step is recorded (nothing is computed during the recording, the commands are only noted); then the recording is replayed for all the remaining steps (section 11).

**Why no rule is broken.** A replayed step does exactly the same arithmetic as a step run directly. The recording contains operations, not results: no weight, no image, no label. It is made again in every run, and making it is counted in the time.

**The one thing that passes from one run to the next** is `RECORDING_KEPT_FROM_THE_PREVIOUS_RUN`. It is kept only for the GPU memory in which the recorded step puts its intermediate results: the new recording uses the same memory again instead of asking for new memory, and the old recording is then released. It is memory only: every number in it is computed again before it is used.

PyTorch also has a shorter way to record (`with torch.cuda.graph(...)`), but it empties the memory caches first, which costs a bit more per run; this is why the recording is started and ended explicitly here.

In [10]:
RECORDING_KEPT_FROM_THE_PREVIOUS_RUN = []


def record_one_training_step_as_a_cuda_graph(run_one_training_step):
    recording = torch.cuda.CUDAGraph()

    memory_pool = None
    for previous_recording in RECORDING_KEPT_FROM_THE_PREVIOUS_RUN:
        memory_pool = previous_recording.pool()

    torch.cuda.synchronize()
    with torch.cuda.stream(torch.cuda.Stream()):
        recording.capture_begin(pool=memory_pool)
        run_one_training_step()
        recording.capture_end()

    for previous_recording in RECORDING_KEPT_FROM_THE_PREVIOUS_RUN:
        previous_recording.reset()
    RECORDING_KEPT_FROM_THE_PREVIOUS_RUN[:] = [recording]

    return recording

## 10. The hard images, and the plan of the steps after the first pass

During the first pass the loss of every train image was stored. `choose_the_hard_images` receives these losses and nothing else. From each of the 156 full batches of the first pass it keeps the 38 images (10%) with the largest loss: 5,928 hard images.

`plan_the_steps_on_the_hard_images` fills in what the remaining 78 steps read: the hard images, several times over, each time in a new random order; and for each of these steps a window drawn at random among the nine (a shift of -1, 0 or +1 pixel in each direction, the same for the whole batch). The random numbers are drawn fresh in every run.

Only losses of train images enter here; no test image and no label of the test set.

In [11]:
def choose_the_hard_images(loss_of_each_train_image):
    images_in_full_batches = FULL_BATCHES_OF_THE_FIRST_PASS * BATCH_SIZE
    losses = loss_of_each_train_image[:images_in_full_batches]
    losses_of_each_batch = losses.view(FULL_BATCHES_OF_THE_FIRST_PASS, BATCH_SIZE)

    hardest = losses_of_each_batch.topk(HARD_IMAGES_KEPT_FROM_EACH_BATCH, dim=1)
    place_in_its_batch = hardest.indices

    first_image_of_each_batch = torch.arange(FULL_BATCHES_OF_THE_FIRST_PASS, device=THE_GPU)
    first_image_of_each_batch = first_image_of_each_batch.view(-1, 1) * BATCH_SIZE
    return (place_in_its_batch + first_image_of_each_batch).reshape(-1)


def plan_the_steps_on_the_hard_images(storage, hard_images):
    places_to_fill = STEPS_ON_THE_HARD_IMAGES * BATCH_SIZE
    passes_over_the_hard_images = math.ceil(places_to_fill / len(hard_images))

    random_numbers = torch.rand(passes_over_the_hard_images, len(hard_images), device=THE_GPU)
    random_orders = random_numbers.argsort(dim=1).view(-1)
    hard_images_in_random_order = hard_images[random_orders[:places_to_fill]]

    random_windows = torch.randint(
        0, NUMBER_OF_WINDOWS, (STEPS_ON_THE_HARD_IMAGES,), device=THE_GPU)

    first = STEPS_OF_THE_FIRST_PASS
    storage.train_images_of_each_step[first:] = hard_images_in_random_order.view(-1, BATCH_SIZE)
    storage.window_of_each_step[first:] = random_windows

## 11. The training: receives ONLY the train images and the train labels (rule 4)

Everything in this function is inside the clock:

1. build the four tables (section 5);
2. create the random initial weights and the empty storage (section 6);
3. convert the train images to half precision with their border (section 4);
4. first pass: all train images in their stored order, not shifted; 2 steps run directly, one step recorded, the rest replayed (section 9);
5. choose the hard images and plan the remaining steps (section 10);
6. replay the recorded step for the remaining steps.

It returns the trained weights.

In [12]:
def train_the_network(train_images, train_labels):
    tables = build_the_tables_from_the_settings_only()
    storage = create_the_random_initial_weights_and_the_empty_storage()
    stored_train_images = store_images_in_half_precision_with_a_blank_border(train_images)

    first_pass = tables.train_images_of_each_first_pass_step
    storage.train_images_of_each_step[:STEPS_OF_THE_FIRST_PASS] = first_pass

    def run_one_training_step():
        one_training_step(storage, tables, stored_train_images, train_labels)

    for _ in range(STEPS_RUN_DIRECTLY_BEFORE_THE_RECORDING):
        run_one_training_step()

    recording = record_one_training_step_as_a_cuda_graph(run_one_training_step)

    for _ in range(STEPS_OF_THE_FIRST_PASS - STEPS_RUN_DIRECTLY_BEFORE_THE_RECORDING):
        recording.replay()

    hard_images = choose_the_hard_images(storage.loss_of_each_train_image)
    plan_the_steps_on_the_hard_images(storage, hard_images)

    for _ in range(STEPS_ON_THE_HARD_IMAGES):
        recording.replay()

    return storage.all_weights.detach()

## 12. The prediction: receives ONLY images and trained weights (rule 6)

There are no labels among the inputs, and the function changes no weight. For each image it does this:

1. **First look.** The image is read through the window without shift; the network gives 10 outputs; the predicted digit is the one with the largest output.
2. **Is the image unsure?** The 10 outputs are turned into probabilities. If the gap between the image's own two largest probabilities is below `GAP_BELOW_WHICH_AN_IMAGE_GETS_A_SECOND_LOOK` (0.5, a fixed setting of section 1), the image is unsure.
3. **Second look, only for an unsure image.** The image is also read through the 8 shifted windows. The log-probabilities of its 9 looks are added, and the predicted digit is the one with the largest sum.

**Why this respects rule 6.** Whether an image gets a second look depends only on that image's own outputs and on a fixed number. There is no ranking of the images, no "the N least certain images", and no statistic of the batch. The function works on many images at once only to compute the same thing for each of them in one go: `[unsure_images]` picks out rows, and every row is computed on its own. So an image gets the same prediction whether it is given alone or together with others.

The number of unsure images changes from run to run (about 270 of the 10,000 test images, by the project's record), and so does the time of the second look; all of it is inside the clock.

The function builds the position lists and converts its images itself, so that its only inputs are the images and the weights.

In [13]:
@torch.no_grad()
def predict_the_digits(images, trained_weights):
    stored_images = store_images_in_half_precision_with_a_blank_border(images)
    pixel_positions_of_each_window = pixel_positions_read_through_each_window()
    weights = trained_weights.half()

    pixel_positions = pixel_positions_of_each_window[WINDOW_WITHOUT_SHIFT]
    patches = stored_images[:, pixel_positions]
    first_look = outputs_of_the_network(patches, weights)
    predicted_digits = first_look.argmax(dim=1)

    two_largest_probabilities = first_look.softmax(dim=1).topk(2, dim=1).values
    gap_of_each_image = two_largest_probabilities[:, 0] - two_largest_probabilities[:, 1]
    is_unsure = gap_of_each_image < GAP_BELOW_WHICH_AN_IMAGE_GETS_A_SECOND_LOOK
    unsure_images = is_unsure.nonzero().view(-1)

    if len(unsure_images) > 0:
        stored_unsure_images = stored_images[unsure_images]
        summed_log_probabilities = first_look[unsure_images].log_softmax(dim=1)

        for window in SHIFTED_WINDOWS:
            pixel_positions = pixel_positions_of_each_window[window]
            patches = stored_unsure_images[:, pixel_positions]
            another_look = outputs_of_the_network(patches, weights)
            summed_log_probabilities += another_look.log_softmax(dim=1)

        predicted_digits[unsure_images] = summed_log_probabilities.argmax(dim=1)

    return predicted_digits

## 13. The accuracy: the only place where the test labels are used (rules 1 and 4)

In [14]:
def accuracy_in_percent(predicted_digits, true_digits):
    fraction_correct = (predicted_digits == true_digits).float().mean()
    return 100 * fraction_correct.item()

## 14. One timed run: where the clock starts and where it stops (rule 3)

**Before the clock.** Python's garbage collector is run and then switched off, so that it cannot interrupt the run. `torch.cuda.synchronize()` waits until the GPU has finished everything it was given before, so that no earlier work falls into the time.

**Inside the clock, three calls:**

1. `train_the_network(train_images, train_labels)`: everything of section 11;
2. `predict_the_digits(test_images, trained_weights)`: converting the 10,000 test images, the first look, and the second look of the unsure images;
3. `accuracy_in_percent(predicted_digits, test_labels)`: comparing with the labels and bringing the number back from the GPU.

Then `torch.cuda.synchronize()` waits until the GPU has really finished, and the clock stops.

**What is measured.** The time is `seconds`: everything between the two readings of the clock. The accuracy is the percentage of the 10,000 test images whose predicted digit equals the label.

In [15]:
def one_timed_run_of_training_and_testing(train_images, train_labels, test_images, test_labels):
    gc.collect()
    gc.disable()
    torch.cuda.synchronize()

    clock_start = time.perf_counter()

    trained_weights = train_the_network(train_images, train_labels)
    predicted_digits = predict_the_digits(test_images, trained_weights)
    accuracy = accuracy_in_percent(predicted_digits, test_labels)
    torch.cuda.synchronize()

    seconds = time.perf_counter() - clock_start

    gc.enable()
    return seconds, accuracy

## 15. The measurement: one warm-up run that is not counted, then 30 runs

Every run starts from new random weights. Nothing is chosen between runs and no run is left out.

In [16]:
for _ in range(NUMBER_OF_WARM_UP_RUNS_NOT_COUNTED):
    one_timed_run_of_training_and_testing(train_images, train_labels, test_images, test_labels)

seconds_of_each_run = []
accuracy_of_each_run = []

for _ in range(NUMBER_OF_TIMED_RUNS):
    seconds, accuracy = one_timed_run_of_training_and_testing(
        train_images, train_labels, test_images, test_labels)
    seconds_of_each_run.append(seconds)
    accuracy_of_each_run.append(accuracy)

The result: the mean of the runs, ± the standard error of the mean (the spread of the runs divided by the square root of their number).

In [17]:
def mean_and_standard_error_of_the_mean(values):
    values = torch.tensor(values, dtype=torch.float64)
    standard_error = values.std() / math.sqrt(len(values))
    return values.mean().item(), standard_error.item()


mean_seconds, error_of_the_seconds = mean_and_standard_error_of_the_mean(seconds_of_each_run)
mean_accuracy, error_of_the_accuracy = mean_and_standard_error_of_the_mean(accuracy_of_each_run)

print(f'{NUMBER_OF_TIMED_RUNS} runs counted, after {NUMBER_OF_WARM_UP_RUNS_NOT_COUNTED} '
      f'warm-up run(s) not counted')
print(f'Total train + test time:            '
      f'{1000 * mean_seconds:.2f} +- {1000 * error_of_the_seconds:.2f} ms')
print(f'Accuracy on the 10,000 test images: '
      f'{mean_accuracy:.3f} +- {error_of_the_accuracy:.3f} %')
print(f'Fastest and slowest single run:     '
      f'{1000 * min(seconds_of_each_run):.2f} and {1000 * max(seconds_of_each_run):.2f} ms')
print(f'Measured on {platform.node()}, {torch.cuda.get_device_name(0)}, '
      f'PyTorch {torch.__version__}')

30 runs counted, after 1 warm-up run(s) not counted
Total train + test time:            231.48 +- 0.36 ms
Accuracy on the 10,000 test images: 99.574 +- 0.007 %
Fastest and slowest single run:     226.18 and 235.28 ms
Measured on 377a6c0af0c7, Tesla T4, PyTorch 2.14.1+cu130
